# Day 2-3: Transformer & MarianMT — 사전학습 모델 활용

**강의 시간**: 1.5시간  
**학습 목표**:
- Transformer의 Self-Attention 메커니즘 이해
- Multi-Head Attention과 Positional Encoding 역할 파악
- Encoder-Decoder 구조의 발전 과정 (RNN → LSTM → Transformer)
- 사전학습 모델(MarianMT)의 위력 체험
- Beam Search를 통한 번역 품질 향상
- HuggingFace Transformers 라이브러리 실전 활용

**사전 요구사항**: Day 2-2 완료  
**예상 성능**: LSTM+Attention ~15-22 BLEU → MarianMT ~35-50 BLEU 🚀

## 🧠 0. Transformer 이론

### 0.1 RNN/LSTM의 근본적 한계

**문제 1: 순차 처리 → 병렬화 불가**
```
RNN/LSTM: h₁ → h₂ → h₃ → h₄ → h₅
           ↓    ↓    ↓    ↓    ↓
        순서대로만 계산 가능 (GPU 활용 ✗)
```

**문제 2: 긴 시퀀스에서 정보 손실**
```
문장: "The movie, which was released in 2010 and won many awards, was great."

h₁₀에서 "The movie"의 정보가 희미해짐
Attention으로 완화했지만 여전히 sequential 처리
```

**Transformer의 해결책:**
```
✅ RNN 완전 제거
✅ Self-Attention으로 모든 단어 간 관계 동시 계산
✅ 완전 병렬 처리 → GPU 100% 활용
✅ 긴 문장도 직접 참조
```

### 0.2 Self-Attention 메커니즘

**핵심 아이디어**: 문장 내 모든 단어가 서로를 직접 바라봄

```
문장: "The cat sat on the mat"

"sat"에 대한 Self-Attention:
  The  : 0.05 (관사, 중요도 낮음)
  cat  : 0.40 ← 주어 (누가 앉았나?)
  sat  : 0.20 (자기 자신)
  on   : 0.10 (전치사)
  the  : 0.05 (관사)
  mat  : 0.20 ← 장소 (어디에 앉았나?)
```

**수식:**
```
1. Q (Query), K (Key), V (Value) 계산
   Q = X · W_q
   K = X · W_k
   V = X · W_v

2. Attention Score 계산
   Attention(Q, K, V) = softmax(Q·K^T / √d_k) · V
```

💡 **왜 √d_k로 나누나요?**  
내적 값이 너무 커지면 softmax가 saturate되어 gradient가 소실됩니다.  
스케일링으로 안정적인 학습을 보장합니다.

### 0.3 Multi-Head Attention

**한 관점이 아니라 여러 관점에서 동시에 보기!**

```
Head 1: 문법 관계 (주어-동사, 수식 관계)
Head 2: 의미 관계 (유사어, 반의어)
Head 3: 위치 정보 (인접성)
Head 4: 대명사 참조
...
Head 8: 종합적 맥락

최종 = Concat(Head₁, ..., Head₈) · W_o
```

**비유:**
- Single-Head: 한 각도에서만 본 사진
- Multi-Head: 8개의 카메라로 동시에 찍은 사진 → 입체적 이해

### 0.4 Positional Encoding

**문제**: Attention은 순서 정보가 없음!
```
"The cat ate the mouse" ≈ "The mouse ate the cat" (Attention 입장에서는 동일)
```

**해결**: 위치 정보를 벡터에 더하기
```python
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

Final = Embedding + Positional Encoding
```

**특징:**
- 학습 없이 고정값 사용 (sin/cos 함수)
- 위치가 멀어질수록 PE 값도 변화
- 상대적 위치 관계도 표현 가능

## 🔧 1. 환경 설정

In [ ]:
# 라이브러리 설치 (약 1-2분 소요)
%pip install -q transformers sacrebleu 'mlflow>=2,<3' dagshub sentencepiece

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# HuggingFace Transformers
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)
import torch
from torch.utils.data import Dataset

# BLEU Score
import sacrebleu

# MLflow & Dagshub
import mlflow
import dagshub

# 시각화 설정
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ Device: {device}")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

# 폰트 파일 경로
font_path = "NanumGothic.ttf"

# 폰트 매니저에 폰트 추가
fm.fontManager.addfont(font_path)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# 폰트 속성 설정
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

print(f"✅ PyTorch version : {torch.__version__}")
print(f"✅ CUDA available  : {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device    : {device}")

🔥 이 부분은 수정이 필요합니다.

**repo_owner**와 **repo_name**을 본인의 Dagshub 정보로 채워 주세요.

In [ ]:
# Dagshub & MLflow 재연동
repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(
    repo_owner=repo_owner,
    repo_name=repo_name,
    mlflow=True
)

mlflow.set_experiment('day2-translation-seq2seq')
print('✅ Dagshub 연동 완료!')


## 📂 2. 데이터 준비

Day 2-1에서 사용한 Tatoeba 데이터를 그대로 활용합니다.  
**이 셀을 Day 2-1에서 복사하거나, 간단히 샘플 데이터로 대체할 수 있습니다.**

In [ ]:
# NLTK punkt tokenizer 다운로드
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ 라이브러리 설치 완료!")

In [ ]:
import os
import pandas as pd
import numpy as np
import kagglehub
import torch
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
import nltk

# NLTK 토큰나이저 데이터 다운로드 (최초 1회)
nltk.download('punkt')

# 1. KaggleHub를 통한 데이터 다운로드 및 경로 설정
# ---------------------------------------------------------------------------
print("📥 KaggleHub를 통해 데이터 다운로드 중...")
download_path = kagglehub.dataset_download("kaushal2896/english-to-german")
files = os.listdir(download_path)

# deu.txt 파일 찾기
target_files = [f for f in files if f.endswith('.txt') and 'deu' in f]
if not target_files:
    raise FileNotFoundError("❌ 데이터를 찾을 수 없습니다.")

DATA_PATH = os.path.join(download_path, target_files[0])
print(f"✅ 실제 데이터 경로: {DATA_PATH}")

# 2. 데이터 로드 및 파싱
# ---------------------------------------------------------------------------
# 탭(\t)으로 구분된 3개 열 (EN, DE, Attribution)
all_df = pd.read_csv(
    DATA_PATH,
    sep='\t',
    header=None,
    names=['english', 'german', 'attribution'],
    encoding='utf-8'
)
all_df = all_df[['english', 'german']] # 필요한 열만 선택
print(f"✅ 전체 문장 쌍 로드: {len(all_df):,}개")

# 3. 필터링 및 샘플링 (사용자 설정 반영)
# ---------------------------------------------------------------------------
NUM_SAMPLES = 5000
MAX_WORDS   = 12
MIN_WORDS   = 3

# 영어 단어 수 기준 필터링
all_df['en_wc'] = all_df['english'].apply(lambda x: len(str(x).split()))
filtered = all_df[
    (all_df['en_wc'] >= MIN_WORDS) &
    (all_df['en_wc'] <= MAX_WORDS)
].copy()

# 지정된 개수만큼 랜덤 샘플링
df = filtered.sample(n=NUM_SAMPLES, random_state=42).reset_index(drop=True)
print(f"✅ 샘플링 완료: {len(df):,}개 (필터 후 {len(filtered):,}개 중)")

# 4. 토큰화 (Tokenization)
# ---------------------------------------------------------------------------
def tokenize(text: str) -> list:
    return word_tokenize(text.lower())

print("🔤 토큰화 진행 중...")
df['en_tokens'] = df['english'].apply(tokenize)
df['de_tokens'] = df['german'].apply(tokenize)

# 5. Train / Validation 데이터 분리 (80/20)
# ---------------------------------------------------------------------------
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"✅ 데이터 분리 완료!")
print(f"   - Train: {len(train_df):,}개")
print(f"   - Val  : {len(val_df):,}개")
print("-" * 30)
print("샘플 확인 (Train):")
print(train_df[['english', 'german']].head())

## 🤖 3. MarianMT 사전학습 모델

### 3.1 MarianMT란?

**Helsinki-NLP의 오픈소스 번역 모델**

**특징:**
- **대규모 사전학습**: OPUS 코퍼스 (수백만~수억 문장)
- **100+ 언어 쌍 지원**: en→de, de→en, en→fr, ko→en 등
- **표준 Transformer**: Encoder 6층 + Decoder 6층
- **HuggingFace 통합**: 3줄 코드로 사용 가능

**왜 처음부터 학습하지 않나요?**
```
처음부터 학습 (From Scratch):
  ❌ 수백만 문장 필요
  ❌ 수일~수주 학습 시간
  ❌ 대규모 GPU 인프라
  ✅ BLEU ~20-30

사전학습 모델 (Pre-trained):
  ✅ 수천 문장으로 Fine-tuning
  ✅ 수시간 학습
  ✅ 단일 GPU로 충분
  ✅ BLEU ~40-60 🚀
```

🔥 이 부분을 같이 작성해봅시다.

사용할 **model_name**을 채워보세요. (예: `'Helsinki-NLP/opus-mt-en-de'`)

In [ ]:
# MarianMT 모델 로드 (영어 → 독일어)
model_name = # 🔥 직접 작성이 필요합니다. (예: 'Helsinki-NLP/opus-mt-en-de')

print('📥 모델 다운로드 중... (처음 실행 시 1-2분 소요)')
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

print(f'✅ MarianMT 로드 완료!')
print(f'   Model: {model_name}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')


## 🔮 4. Zero-Shot 번역 (학습 없이 바로 사용)

🔥 이 부분을 같이 작성해봅시다.

**translate()** 함수에서 `model.generate()`로 번역을 생성하고, `tokenizer.batch_decode()`로 디코딩하는 부분을 완성해 보세요.

In [ ]:
def translate(text, beam_size=5, max_length=128):
    """
    MarianMT로 번역
    Args:
        text: 영어 문장 (str or list of str)
        beam_size: Beam Search width (1=greedy, 5=권장)
        max_length: 최대 생성 길이
    Returns:
        translated: 독일어 번역 (str or list)
    """
    model.eval()
    is_single = isinstance(text, str)
    texts = [text] if is_single else text

    # Tokenize
    inputs = tokenizer(
        texts, return_tensors='pt', padding=True,
        truncation=True, max_length=max_length
    ).to(device)

    # Generate
    with torch.no_grad():
        outputs = # 🔥 직접 작성이 필요합니다. (model.generate(**inputs, num_beams=beam_size, ...))

    # Decode
    translations = # 🔥 직접 작성이 필요합니다. (tokenizer.batch_decode(outputs, skip_special_tokens=True))

    return translations[0] if is_single else translations

print('✅ 번역 함수 정의 완료!')


In [ ]:
# Zero-Shot 번역 테스트
test_sentences = [
    "I love you",
    "The cat is black",
    "How are you today",
    "Machine learning is amazing",
    "I am learning German at university"
]

print("=" * 80)
print("Zero-Shot Translation (학습 없이 바로 사용)")
print("=" * 80)

for en in test_sentences:
    de = translate(en, beam_size=5)
    print(f"EN: {en}")
    print(f"DE: {de}")
    print()

## 🔍 5. Beam Search 실험

### 5.1 Greedy vs Beam Search

**Greedy Decoding (beam_size=1):**
```
매 스텝마다 가장 확률 높은 단어만 선택
→ 빠르지만 최적이 아닐 수 있음
```

**Beam Search (beam_size=5):**
```
매 스텝마다 상위 5개 후보 유지
→ 느리지만 더 나은 번역
```

**예시:**
```
Input: "I love you"

Step 1: <SOS> → ["Ich"(0.6), "I"(0.3), "Ik"(0.1)]

Step 2:
  "Ich" → ["liebe"(0.5), "mag"(0.4)]
  "I"   → ["love"(0.6)]

Beams:
  "Ich liebe" (0.6×0.5 = 0.30)
  "Ich mag"   (0.6×0.4 = 0.24)
  "I love"    (0.3×0.6 = 0.18)

Final: "Ich liebe dich" (최고 확률 경로)
```

In [ ]:
# Beam Size에 따른 번역 품질 비교
beam_sizes = [1, 3, 5, 10]
test_text = "I love machine learning and artificial intelligence"

print(f"Input: {test_text}")
print()
print("=" * 70)
print(f"{'Beam Size':<12} {'Translation':<50} {'Time (ms)':>8}")
print("=" * 70)

import time
results = []

for bs in beam_sizes:
    start = time.time()
    translation = translate(test_text, beam_size=bs)
    elapsed = (time.time() - start) * 1000

    results.append({'beam_size': bs, 'translation': translation, 'time_ms': elapsed})
    print(f"{bs:<12} {translation:<50} {elapsed:>7.1f}")

print("=" * 70)
print()
print("💡 관찰:")
print("  - Beam Size ↑ → 품질 ↑ (어느 시점까지)")
print("  - Beam Size ↑ → 속도 ↓")
print("  - 실무 권장: beam_size=5 (품질/속도 균형)")

## 📊 6. BLEU Score 평가

In [ ]:
def calculate_bleu_marianmt(model, df, beam_size=5, max_samples=500):
    """
    MarianMT 모델의 BLEU Score 계산

    Returns:
        bleu_score (float)
    """
    references = []
    hypotheses = []

    n_samples = min(max_samples, len(df))

    for i in tqdm(range(n_samples), desc="Calculating BLEU"):
        en = df.iloc[i]['english']
        de_ref = df.iloc[i]['german']

        de_pred = translate(en, beam_size=beam_size)

        references.append(de_ref)
        hypotheses.append(de_pred)

    bleu = sacrebleu.corpus_bleu(hypotheses, [references], force=True)
    return bleu.score

print("✅ BLEU 계산 함수 정의 완료!")

In [ ]:
# MarianMT Zero-Shot BLEU 평가
print("📊 Zero-Shot BLEU Score 계산 중...")
print("   (Val set 200개 샘플, beam_size=5)")
print()

bleu_zero_shot = calculate_bleu_marianmt(
    model, val_df, beam_size=5, max_samples=200
)

print()
print("=" * 60)
print(f"  MarianMT Zero-Shot BLEU: {bleu_zero_shot:.2f}")
print("=" * 60)
print()
print("💡 Day 2-2 LSTM+Attention과 비교:")
print(f"   LSTM Baseline   : ~8-15 BLEU")
print(f"   LSTM+Attention  : ~15-22 BLEU")
print(f"   MarianMT        : {bleu_zero_shot:.2f} BLEU 🚀")
print()
print("   → 사전학습의 위력!")

## 📈 7. MLflow 실험 기록

🔥 이 부분은 수정이 필요합니다.

**run_name**을 실험을 구분하기 쉬운 이름으로 채운 뒤 실행하고, Dagshub UI에서 Day 2 전체 실험을 비교해 보세요.

In [ ]:
# 이전 MLflow run 종료 (혹시 남아있을 경우)
try:
    mlflow.end_run()
except:
    pass

# MLflow Run 시작
with mlflow.start_run(run_name=""):  # 🔥 직접 작성이 필요합니다.

    # 파라미터 로깅
    mlflow.log_params({
        'model'         : 'MarianMT',
        'model_name'    : model_name,
        'architecture'  : 'Transformer (Encoder-Decoder)',
        'encoder_layers': 6,
        'decoder_layers': 6,
        'pretrained'    : True,
        'fine_tuned'    : False,
        'beam_size'     : 5,
        'max_length'    : 128,
    })

    # 메트릭 로깅
    mlflow.log_metric('val_bleu', bleu_zero_shot)

    # 샘플 번역 저장
    sample_indices = [0, 10, 20, 50, 100]
    with open('translations_marianmt.txt', 'w', encoding='utf-8') as f:
        f.write("MarianMT Zero-Shot — 샘플 번역\n")
        f.write("=" * 60 + "\n\n")

        for idx in sample_indices:
            if idx >= len(val_df):
                continue

            en = val_df.iloc[idx]['english']
            de_ref = val_df.iloc[idx]['german']
            de_pred = translate(en, beam_size=5)

            f.write(f"[Sample {idx}]\n")
            f.write(f"  EN (Source)    : {en}\n")
            f.write(f"  DE (Reference) : {de_ref}\n")
            f.write(f"  DE (Predicted) : {de_pred}\n\n")

    mlflow.log_artifact('translations_marianmt.txt')

    run_id = mlflow.active_run().info.run_id
    print(f"✅ MLflow Run ID: {run_id}")

## 📊 8. 모델 성능 비교

In [ ]:
# Day 2 전체 모델 성능 비교
models = [
    {'name': 'LSTM Baseline', 'bleu': 10.5, 'params': '~2M'},
    {'name': 'LSTM + Attention', 'bleu': 18.2, 'params': '~2.5M'},
    {'name': 'MarianMT (Zero-Shot)', 'bleu': bleu_zero_shot, 'params': '~74M'},
]

# 비교 테이블
comparison_df = pd.DataFrame(models)

print("=" * 70)
print("  Day 2 모델 성능 종합 비교")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BLEU 비교
colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = axes[0].barh(comparison_df['name'], comparison_df['bleu'], color=colors)
axes[0].set_xlabel('BLEU Score', fontweight='bold')
axes[0].set_title('Translation Quality Comparison', fontweight='bold')
axes[0].set_xlim(0, max(comparison_df['bleu']) * 1.1)

# 값 표시
for bar, bleu in zip(bars, comparison_df['bleu']):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{bleu:.1f}', va='center', fontweight='bold')

# 상대 성능 향상
baseline_bleu = comparison_df.iloc[0]['bleu']
improvements = [(bleu - baseline_bleu) / baseline_bleu * 100
                for bleu in comparison_df['bleu']]

bars2 = axes[1].bar(comparison_df['name'], improvements, color=colors)
axes[1].set_ylabel('Improvement vs Baseline (%)', fontweight='bold')
axes[1].set_title('Relative Performance Gain', fontweight='bold')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_ylim(-10, max(improvements) * 1.1)

# 값 표시
for bar, imp in zip(bars2, improvements):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{imp:+.0f}%', ha='center', fontweight='bold')

plt.suptitle('Day 2: Seq2Seq → LSTM → Transformer 성능 진화',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('day2_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

mlflow.log_artifact('day2_model_comparison.png')

## 🧠 9. 핵심 개념 정리

### 오늘 배운 것

**Transformer의 혁신**
- RNN 완전 제거 → 순차 처리에서 벗어남
- Self-Attention → 모든 단어 간 직접 연결
- 병렬 처리 → GPU 100% 활용
- Positional Encoding → 순서 정보 보존

**Self-Attention**
```
Q (Query): "이 단어가 찾는 것은?"
K (Key):   "나는 이런 특징이야"
V (Value): "내 실제 정보는 이거야"

Attention = softmax(Q·K^T / √d_k) · V
```

**Multi-Head Attention**
- 8개의 다른 관점에서 동시에 보기
- Head 1: 문법, Head 2: 의미, Head 3: 위치...
- 더 풍부한 표현력

**사전학습 모델의 위력**
```
처음부터 학습: BLEU ~20-30, 수일 소요
MarianMT:      BLEU ~40-60, 즉시 사용 🚀
```

**Beam Search**
- Greedy (beam=1): 빠르지만 최적 아님
- Beam (beam=5): 품질↑, 속도 적절
- 실무 권장: beam_size=5

---

### Day 2 전체 요약

**Day 2-1: 데이터 준비**
- 병렬 코퍼스, Vocabulary, BLEU Score

**Day 2-2: LSTM Seq2Seq**
- Encoder-Decoder, Teacher Forcing
- Attention 추가 → BLEU +40%

**Day 2-3: Transformer**
- Self-Attention, 병렬 처리
- 사전학습 모델 → BLEU +150%

**성능 진화:**
```
LSTM Baseline     : ~10 BLEU
LSTM + Attention  : ~18 BLEU (+80%)
MarianMT          : ~40 BLEU (+300%) 🚀
```

## ✅ Day 2-3 완료 체크리스트

- [ ] Self-Attention 메커니즘 이해 (Q, K, V)
- [ ] Multi-Head Attention의 역할 파악
- [ ] Positional Encoding의 필요성 이해
- [ ] RNN/LSTM vs Transformer 차이 비교
- [ ] MarianMT 모델 로드 및 사용
- [ ] Zero-Shot 번역 실행
- [ ] Beam Search 원리 및 실험
- [ ] BLEU Score 계산 및 비교
- [ ] MLflow에 실험 기록
- [ ] Day 2 전체 모델 성능 비교
- [ ] HuggingFace Transformers 라이브러리 활용

## 🎯 다음 단계

### Day 3 예고: 실전 이미지 분류

**MNIST 손글씨 인식 (Kaggle 제출)**
- CNN 기초: Conv → Pool → FC
- Data Augmentation
- Kaggle Competition 참여

**의료 이미지 분류 (ChestX-ray8)**
- Transfer Learning (ResNet, EfficientNet)
- Class Imbalance 처리
- Multi-Label Classification

**예상 결과:**
```
MNIST: ~99% Accuracy
ChestX-ray8: ~75-80% AUC
```

---

### 축하합니다! 🎉

Day 2를 완료했습니다!

**달성한 것:**
- ✅ Seq2Seq 모델 이해 및 구현
- ✅ LSTM의 장기 기억 메커니즘
- ✅ Attention으로 성능 40% 향상
- ✅ Transformer의 Self-Attention
- ✅ 사전학습 모델로 BLEU 3배 향상
- ✅ HuggingFace 라이브러리 실전 활용

이제 여러분은 최신 NLP 기술의 핵심을 이해했습니다!

수고하셨습니다! 🚀